# 08 - Análisis de Stock: CatBoost vs Naive

Este notebook implementa el análisis de costes de inventario comparando:
- **Modelo CatBoost**: Predicciones del mejor modelo ML
- **Modelo Naive**: Media móvil de 7 días

Se utiliza la fórmula de Yamazaki (2015) para calcular el stock de seguridad:
$$SS = Z \times \sigma \times \sqrt{L + 1}$$

Donde:
- $Z$: Factor de servicio (1.645 para 95%, 2.326 para 99%)
- $\sigma$: Desviación estándar del error (RMSE)
- $L$: Lead time (días de aprovisionamiento)

**Referencia**: Yamazaki, Y. (2015). DOI: https://doi.org/10.1080/00207543.2015.1076179

## 1. Configuración e Importaciones

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Parámetros de costes (ajustables según negocio)
TASA_ALMACENAMIENTO_ANUAL = 0.20  # 20% del valor del producto
COSTE_RUPTURA_UNITARIO = 0.10     # 10% del precio como penalización
NIVEL_SERVICIO = 0.95             # 95% de nivel de servicio
Z_SCORE = 1.645                   # Factor Z para 95% de confianza

print("Configuración cargada")
print(f"Nivel de servicio: {NIVEL_SERVICIO*100}%")
print(f"Z-score: {Z_SCORE}")
print(f"Tasa almacenamiento anual: {TASA_ALMACENAMIENTO_ANUAL*100}%")
print(f"Coste ruptura unitario: {COSTE_RUPTURA_UNITARIO*100}%")

## 2. Carga de Datos

In [ ]:
# Cargar predicciones de CatBoost
df_test = pd.read_csv('../datos/df_test_catboost.csv', sep=';', decimal=',')
print(f"Datos de test cargados: {df_test.shape}")
print(f"Columnas: {df_test.columns.tolist()}")

# Cargar datos de ciclo de aprovisionamiento
df_ciclos = pd.read_csv('../datos/DatosCicloAprovisionamiento.csv', sep=';', decimal=',')
print(f"\nDatos de ciclos: {df_ciclos.shape}")
print(f"Columnas: {df_ciclos.columns.tolist()}")

# Cargar precios medios
df_precios = pd.read_csv('../datos/DatosPrecioMedio.csv', sep=';', decimal=',')
# Convertir precio de formato europeo (coma) a float
if df_precios['eurPrecioMedio'].dtype == 'object':
    df_precios['eurPrecioMedio'] = df_precios['eurPrecioMedio'].str.replace(',', '.').astype(float)
print(f"\nDatos de precios: {df_precios.shape}")
print(f"Columnas: {df_precios.columns.tolist()}")

# Vista previa
print("\nPrimeras filas de df_test:")
display(df_test.head())
print("\nPrimeras filas de df_ciclos:")
display(df_ciclos.head())
print("\nPrimeras filas de df_precios:")
display(df_precios.head())

## 3. Cálculo de RMSE para CatBoost

In [ ]:
# Calcular RMSE por producto para CatBoost
rmse_catboost = df_test.groupby('producto').apply(
    lambda x: np.sqrt(mean_squared_error(x['udsVenta'], x['udsVentaPred']))
).reset_index(name='rmse_catboost')

print(f"RMSE calculado para {len(rmse_catboost)} productos")
print(f"\nEstadísticas de RMSE CatBoost:")
print(rmse_catboost['rmse_catboost'].describe())
print(f"\nPrimeros productos:")
display(rmse_catboost.head(10))

## 4. Cálculo de RMSE para Modelo Naive (Media Móvil 7 días)

In [ ]:
# Ordenar por producto y secuencia para rolling window
df_test_sorted = df_test.sort_values(['producto', 'idSecuencia']).copy()

# Calcular predicción naive: media móvil de 7 días desplazada 1 período
df_test_sorted['udsVentaPred_naive'] = df_test_sorted.groupby('producto')['udsVenta'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean().shift(1)
)

# Para el primer valor de cada producto, usar la media global del producto
df_test_sorted['udsVentaPred_naive'] = df_test_sorted.groupby('producto')['udsVentaPred_naive'].transform(
    lambda x: x.fillna(df_test_sorted.loc[x.index, 'udsVenta'].mean())
)

# Calcular RMSE por producto para Naive
rmse_naive = df_test_sorted.groupby('producto').apply(
    lambda x: np.sqrt(mean_squared_error(x['udsVenta'], x['udsVentaPred_naive']))
).reset_index(name='rmse_naive')

print(f"RMSE Naive calculado para {len(rmse_naive)} productos")
print(f"\nEstadísticas de RMSE Naive:")
print(rmse_naive['rmse_naive'].describe())
print(f"\nPrimeros productos:")
display(rmse_naive.head(10))

## 5. Cálculo de Stock de Seguridad (Fórmula de Yamazaki)

In [ ]:
# Fusionar todos los datos
df_analisis = rmse_catboost.merge(rmse_naive, on='producto')
df_analisis = df_analisis.merge(df_ciclos, on='producto')
df_analisis = df_analisis.merge(df_precios, on='producto')

print(f"Datos fusionados: {df_analisis.shape}")

# Calcular stock de seguridad según fórmula de Yamazaki
# SS = Z × σ × √(L + 1)
df_analisis['ss_catboost'] = Z_SCORE * df_analisis['rmse_catboost'] * np.sqrt(df_analisis['diasLeadtime'] + 1)
df_analisis['ss_naive'] = Z_SCORE * df_analisis['rmse_naive'] * np.sqrt(df_analisis['diasLeadtime'] + 1)

print("\nStock de seguridad calculado")
print(f"\nEstadísticas SS CatBoost:")
print(df_analisis['ss_catboost'].describe())
print(f"\nEstadísticas SS Naive:")
print(df_analisis['ss_naive'].describe())

# Diferencia en stock de seguridad
df_analisis['ss_diferencia'] = df_analisis['ss_naive'] - df_analisis['ss_catboost']
df_analisis['ss_reduccion_pct'] = (df_analisis['ss_diferencia'] / df_analisis['ss_naive']) * 100

print(f"\nReducción media en SS: {df_analisis['ss_reduccion_pct'].mean():.2f}%")
display(df_analisis[['producto', 'rmse_catboost', 'rmse_naive', 'ss_catboost', 'ss_naive', 'ss_reduccion_pct']].head(10))

## 6. Cálculo de Stock Medio y Stock Máximo

In [ ]:
# Calcular demanda media por producto
demanda_media = df_test.groupby('producto')['udsVenta'].mean().reset_index(name='demanda_media_diaria')
df_analisis = df_analisis.merge(demanda_media, on='producto')

# Stock de ciclo = (Demanda media × Días entre pedidos) / 2
df_analisis['stock_ciclo'] = (df_analisis['demanda_media_diaria'] * df_analisis['diasEntrePedidos']) / 2

# Stock medio = Stock de ciclo + Stock de seguridad
df_analisis['stock_medio_catboost'] = df_analisis['stock_ciclo'] + df_analisis['ss_catboost']
df_analisis['stock_medio_naive'] = df_analisis['stock_ciclo'] + df_analisis['ss_naive']

# Stock máximo = Demanda media × (Leadtime + Días entre pedidos) + Stock de seguridad
df_analisis['stock_max_catboost'] = df_analisis['demanda_media_diaria'] * (
    df_analisis['diasLeadtime'] + df_analisis['diasEntrePedidos']
) + df_analisis['ss_catboost']

df_analisis['stock_max_naive'] = df_analisis['demanda_media_diaria'] * (
    df_analisis['diasLeadtime'] + df_analisis['diasEntrePedidos']
) + df_analisis['ss_naive']

print("Stock medio y máximo calculados")
print(f"\nStock medio CatBoost: {df_analisis['stock_medio_catboost'].sum():.0f} unidades")
print(f"Stock medio Naive: {df_analisis['stock_medio_naive'].sum():.0f} unidades")
print(f"\nReducción total en stock medio: {(df_analisis['stock_medio_naive'].sum() - df_analisis['stock_medio_catboost'].sum()):.0f} unidades")

display(df_analisis[['producto', 'stock_ciclo', 'stock_medio_catboost', 'stock_medio_naive', 'stock_max_catboost', 'stock_max_naive']].head(10))

## 7. Cálculo de Costes

In [ ]:
# Coste de almacenamiento anual = Stock medio × Precio × Tasa de almacenamiento
df_analisis['coste_almacen_catboost'] = df_analisis['stock_medio_catboost'] * df_analisis['eurPrecioMedio'] * TASA_ALMACENAMIENTO_ANUAL
df_analisis['coste_almacen_naive'] = df_analisis['stock_medio_naive'] * df_analisis['eurPrecioMedio'] * TASA_ALMACENAMIENTO_ANUAL

# Estimar roturas de stock basadas en nivel de servicio no cubierto
# Asumimos que el error no cubierto genera rupturas proporcionales
dias_test = df_test.groupby('producto')['idSecuencia'].nunique().reset_index(name='dias_observados')
df_analisis = df_analisis.merge(dias_test, on='producto')

# Coste de ruptura = RMSE × Días × Precio × Coste ruptura unitario
df_analisis['coste_ruptura_catboost'] = df_analisis['rmse_catboost'] * df_analisis['dias_observados'] * df_analisis['eurPrecioMedio'] * COSTE_RUPTURA_UNITARIO
df_analisis['coste_ruptura_naive'] = df_analisis['rmse_naive'] * df_analisis['dias_observados'] * df_analisis['eurPrecioMedio'] * COSTE_RUPTURA_UNITARIO

# Coste total
df_analisis['coste_total_catboost'] = df_analisis['coste_almacen_catboost'] + df_analisis['coste_ruptura_catboost']
df_analisis['coste_total_naive'] = df_analisis['coste_almacen_naive'] + df_analisis['coste_ruptura_naive']

# Ahorro
df_analisis['ahorro_total'] = df_analisis['coste_total_naive'] - df_analisis['coste_total_catboost']
df_analisis['ahorro_pct'] = (df_analisis['ahorro_total'] / df_analisis['coste_total_naive']) * 100

print("="*80)
print("RESUMEN DE COSTES ANUALES")
print("="*80)
print(f"\n{'MODELO CATBOOST':^40}")
print("-"*80)
print(f"  Coste almacenamiento: {df_analisis['coste_almacen_catboost'].sum():>15,.2f} €")
print(f"  Coste ruptura stock:  {df_analisis['coste_ruptura_catboost'].sum():>15,.2f} €")
print(f"  COSTE TOTAL:          {df_analisis['coste_total_catboost'].sum():>15,.2f} €")

print(f"\n{'MODELO NAIVE':^40}")
print("-"*80)
print(f"  Coste almacenamiento: {df_analisis['coste_almacen_naive'].sum():>15,.2f} €")
print(f"  Coste ruptura stock:  {df_analisis['coste_ruptura_naive'].sum():>15,.2f} €")
print(f"  COSTE TOTAL:          {df_analisis['coste_total_naive'].sum():>15,.2f} €")

print(f"\n{'AHORRO CON CATBOOST':^40}")
print("="*80)
print(f"  Ahorro total:         {df_analisis['ahorro_total'].sum():>15,.2f} €")
print(f"  Reducción:            {(df_analisis['ahorro_total'].sum() / df_analisis['coste_total_naive'].sum() * 100):>15,.2f} %")
print("="*80)

display(df_analisis[['producto', 'coste_total_catboost', 'coste_total_naive', 'ahorro_total', 'ahorro_pct']].head(10))

## 8. Exportar Resultados

In [ ]:
# Seleccionar columnas relevantes para exportar
columnas_exportar = [
    'producto',
    'eurPrecioMedio',
    'diasEntrePedidos',
    'diasLeadtime',
    'demanda_media_diaria',
    'rmse_catboost',
    'rmse_naive',
    'ss_catboost',
    'ss_naive',
    'ss_reduccion_pct',
    'stock_medio_catboost',
    'stock_medio_naive',
    'stock_max_catboost',
    'stock_max_naive',
    'coste_almacen_catboost',
    'coste_almacen_naive',
    'coste_ruptura_catboost',
    'coste_ruptura_naive',
    'coste_total_catboost',
    'coste_total_naive',
    'ahorro_total',
    'ahorro_pct'
]

df_exportar = df_analisis[columnas_exportar].copy()

# Ordenar por ahorro total descendente
df_exportar = df_exportar.sort_values('ahorro_total', ascending=False)

# Exportar a CSV
ruta_salida = '../datos/analisis_stock_comparativo.csv'
df_exportar.to_csv(ruta_salida, sep=';', decimal=',', index=False)

print(f"Resultados exportados a: {ruta_salida}")
print(f"Total de productos analizados: {len(df_exportar)}")
print(f"\nProductos con mayor ahorro:")
display(df_exportar[['producto', 'ahorro_total', 'ahorro_pct']].head(10))

## 9. Resumen Ejecutivo

In [ ]:
# Crear resumen ejecutivo
print("="*80)
print("RESUMEN EJECUTIVO: ANÁLISIS DE INVENTARIO")
print("="*80)
print(f"\nModelos comparados:")
print(f"  - CatBoost (modelo ML optimizado)")
print(f"  - Naive (media móvil 7 días)")
print(f"\nParámetros:")
print(f"  - Nivel de servicio: {NIVEL_SERVICIO*100}%")
print(f"  - Z-score: {Z_SCORE}")
print(f"  - Tasa almacenamiento: {TASA_ALMACENAMIENTO_ANUAL*100}% anual")
print(f"  - Coste ruptura: {COSTE_RUPTURA_UNITARIO*100}% del precio")

print(f"\nProductos analizados: {len(df_analisis)}")
print(f"\nMejora en precisión (RMSE):")
rmse_mejora = ((df_analisis['rmse_naive'].mean() - df_analisis['rmse_catboost'].mean()) / df_analisis['rmse_naive'].mean()) * 100
print(f"  - RMSE medio Naive:     {df_analisis['rmse_naive'].mean():.2f} unidades")
print(f"  - RMSE medio CatBoost:  {df_analisis['rmse_catboost'].mean():.2f} unidades")
print(f"  - Mejora:               {rmse_mejora:.2f}%")

print(f"\nReducción en stock de seguridad:")
ss_total_naive = df_analisis['ss_naive'].sum()
ss_total_catboost = df_analisis['ss_catboost'].sum()
ss_reduccion = ((ss_total_naive - ss_total_catboost) / ss_total_naive) * 100
print(f"  - SS total Naive:       {ss_total_naive:,.0f} unidades")
print(f"  - SS total CatBoost:    {ss_total_catboost:,.0f} unidades")
print(f"  - Reducción:            {ss_reduccion:.2f}%")

print(f"\nReducción en stock medio:")
stock_medio_naive = df_analisis['stock_medio_naive'].sum()
stock_medio_catboost = df_analisis['stock_medio_catboost'].sum()
stock_reduccion = ((stock_medio_naive - stock_medio_catboost) / stock_medio_naive) * 100
print(f"  - Stock medio Naive:    {stock_medio_naive:,.0f} unidades")
print(f"  - Stock medio CatBoost: {stock_medio_catboost:,.0f} unidades")
print(f"  - Reducción:            {stock_reduccion:.2f}%")

print(f"\nImpacto económico anual:")
coste_total_naive = df_analisis['coste_total_naive'].sum()
coste_total_catboost = df_analisis['coste_total_catboost'].sum()
ahorro_total = df_analisis['ahorro_total'].sum()
ahorro_pct = (ahorro_total / coste_total_naive) * 100
print(f"  - Coste total Naive:    {coste_total_naive:>15,.2f} €")
print(f"  - Coste total CatBoost: {coste_total_catboost:>15,.2f} €")
print(f"  - AHORRO ANUAL:         {ahorro_total:>15,.2f} €")
print(f"  - Reducción de coste:   {ahorro_pct:>15,.2f} %")

# Productos con mayor impacto
top_10_ahorro = df_analisis.nlargest(10, 'ahorro_total')
ahorro_top10 = top_10_ahorro['ahorro_total'].sum()
print(f"\nTop 10 productos representan:")
print(f"  - Ahorro: {ahorro_top10:,.2f} € ({(ahorro_top10/ahorro_total)*100:.1f}% del total)")

print(f"\n" + "="*80)
print(f"CONCLUSIÓN: El modelo CatBoost reduce los costes de inventario en {ahorro_pct:.2f}%")
print(f"generando un ahorro anual de {ahorro_total:,.2f} € mediante predicciones más precisas.")
print("="*80)